# 01_Command Recognition (GMM_HMM)

## Objective
The objective of this notebook is to build and compare GMM and HMM models for command recognition.

## Method
Feature extraction plus probabilistic sequence/model comparison (GMM vs HMM).

## Input
Command audio samples and extracted acoustic features.

## Processing Tasks

1. 1️⃣ Setup & Imports.
2. 2️⃣ Dataset Inspection.
3. 3️⃣ Feature Extraction — Frame-Level MFCCs.
4. 4️⃣ GMM Training.
5. 5️⃣ HMM Training.
6. 6️⃣ Evaluation.
7. 📊 Metrics for Biometric Command Recognition.
8. 🔤 Word Error Rate (WER).
9. 7️⃣ Permission-Based Command System.
10. 8️⃣ End-to-End Demo.
11. 9️⃣ Summary.
12. Key Takeaways:.

## Output
- Trained GMM and HMM models.
- Performance comparison metrics and plots.
- Interpretation visuals for model behavior.

## Purpose
Identify the more suitable command recognition model for application integration.


## 1. 1️⃣ Setup & Imports


In [ ]:
import os
import sys
import warnings
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import joblib
import soundfile as sf
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, ConfusionMatrixDisplay)
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')

# ─── Project Paths ────────────────────────────────────────────────────────────
BASE_DIR     = r'E:\Multimodal Biometric Verification System'
COMMANDS_DIR = os.path.join(BASE_DIR, 'Dataset', 'Raw', 'Command Recogntion')
MODELS_DIR   = os.path.join(BASE_DIR, 'Models', 'Command_Recognition')
AUDIO_EXTENSIONS = ('.wav',)
ffmpeg_bin = os.path.join(os.path.dirname(sys.executable), 'Library', 'bin')
if os.path.isdir(ffmpeg_bin):
    os.environ['PATH'] = ffmpeg_bin + os.pathsep + os.environ.get('PATH', '')

# Add src to path
sys.path.insert(0, BASE_DIR)

print('✅ Imports OK')
print(f'   Commands dataset : {COMMANDS_DIR}')
print(f'   Models directory : {MODELS_DIR}')

# Check hmmlearn
try:
    from hmmlearn import hmm
    print('   hmmlearn         : ✅ Available')
    HMM_AVAILABLE = True
except ImportError:
    print('   hmmlearn         : ❌ Not installed (pip install hmmlearn)')
    HMM_AVAILABLE = False

## 2. 2️⃣ Dataset Inspection


In [ ]:
COMMANDS = ['open', 'status']

print('=' * 50)
print('  Command Audio Dataset Overview')
print('=' * 50)

dataset_info = {}

for cmd in COMMANDS:
    cmd_path = os.path.join(COMMANDS_DIR, cmd)
    if not os.path.exists(cmd_path):
        print(f'  [{cmd}] ❌ Folder not found: {cmd_path}')
        dataset_info[cmd] = []
        continue

    wav_files = sorted([f for f in os.listdir(cmd_path) if f.lower().endswith(AUDIO_EXTENSIONS)])
    dataset_info[cmd] = wav_files

    durations = []
    for wf in wav_files:
        y, sr = librosa.load(os.path.join(cmd_path, wf), sr=16000)
        durations.append(len(y) / sr)

    if durations:
        print(f'  [{cmd}] {len(wav_files)} files | '
              f'avg {np.mean(durations):.2f}s | '
              f'min {np.min(durations):.2f}s | '
              f'max {np.max(durations):.2f}s')
    else:
        print(f'  [{cmd}] 0 files')

print('=' * 50)
total = sum(len(v) for v in dataset_info.values())
print(f'  Total: {total} audio files')

In [ ]:
# ─── Visualize one sample per command ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
fig.suptitle('Command Audio Samples — Waveform & Spectrogram', fontsize=14, fontweight='bold')

colors = {'open': '#2196F3', 'status': '#4CAF50'}

for idx, cmd in enumerate(COMMANDS):
    files = dataset_info.get(cmd, [])
    if not files:
        for ax in axes[idx]:
            ax.text(0.5, 0.5, f'No files for "{cmd}"',
                    ha='center', va='center', transform=ax.transAxes)
        continue

    sample_path = os.path.join(COMMANDS_DIR, cmd, files[0])
    y, sr = librosa.load(sample_path, sr=16000)
    t = np.linspace(0, len(y)/sr, len(y))

    # Waveform
    axes[idx, 0].plot(t, y, color=colors[cmd], linewidth=0.8)
    axes[idx, 0].set_title(f'Waveform: "{cmd}"')
    axes[idx, 0].set_xlabel('Time (s)')
    axes[idx, 0].set_ylabel('Amplitude')
    axes[idx, 0].grid(True, alpha=0.3)

    # Mel Spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=sr, hop_length=160,
                                    x_axis='time', y_axis='mel',
                                    ax=axes[idx, 1], cmap='magma')
    axes[idx, 1].set_title(f'Mel Spectrogram: "{cmd}"')
    fig.colorbar(img, ax=axes[idx, 1], format='%+2.0f dB')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'Notebooks', 'Command_Recognition', 'waveforms.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 3. 3️⃣ Feature Extraction — Frame-Level MFCCs

For GMM/HMM, we need **frame-level features**, not averaged features.

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| Coefficients | 13 MFCCs | Standard for speech recognition |
| + Delta | 13 | 1st-order time derivative (speed) |
| + Delta-Delta | 13 | 2nd-order time derivative (acceleration) |
| **Total** | **39D** | Per frame feature vector |
| Frame length | 25ms (400 samples @ 16kHz) | Standard ASR window |
| Frame shift | 10ms (160 samples @ 16kHz) | 60% overlap |


In [ ]:
# ─── Feature Extraction Parameters ───────────────────────────────────────────
SAMPLE_RATE = 16000
N_MFCC      = 13
HOP_LENGTH  = 160    # 10ms
WIN_LENGTH  = 400    # 25ms


def extract_mfcc_sequence(audio_path=None, audio_array=None):
    """Extract frame-level MFCC + Delta + Delta-Delta (39D per frame)."""
    if audio_path is not None:
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    else:
        y = audio_array.astype(np.float32)
        sr = SAMPLE_RATE

    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) < WIN_LENGTH:
        y = np.pad(y, (0, WIN_LENGTH - len(y)))

    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                   hop_length=HOP_LENGTH, n_fft=WIN_LENGTH)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    return np.vstack([mfcc, delta, delta2]).T.astype(np.float32)


# ─── Extract all features ─────────────────────────────────────────────────────
print('Extracting frame-level MFCC features...')

all_features   = {}   # {cmd: list of (n_frames, 39) arrays}
frame_counts   = {}   # {cmd: total frames}
sample_counts  = {}   # {cmd: number of recordings}

for cmd in COMMANDS:
    cmd_path  = os.path.join(COMMANDS_DIR, cmd)
    wav_files = sorted([f for f in os.listdir(cmd_path) if f.lower().endswith(AUDIO_EXTENSIONS)])

    seqs = []
    for wf in wav_files:
        try:
            feats = extract_mfcc_sequence(os.path.join(cmd_path, wf))
            seqs.append(feats)
        except Exception as e:
            print(f'  Skipped {wf}: {e}')

    all_features[cmd]  = seqs
    frame_counts[cmd]  = sum(len(s) for s in seqs)
    sample_counts[cmd] = len(seqs)

    print(f'  [{cmd}] {len(seqs)} recordings | '
          f'{frame_counts[cmd]} total frames | '
          f'shape per sample: {seqs[0].shape if seqs else "N/A"}')

print('\n✅ Feature extraction complete.')
print(f'   Feature dimension: 39 (13 MFCC + 13 Δ + 13 ΔΔ)')

In [ ]:
# ─── Visualize MFCC for one sample per command ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('MFCC Feature Matrix (39 coefficients × frames)', fontsize=13, fontweight='bold')

for idx, cmd in enumerate(COMMANDS):
    if not all_features[cmd]:
        continue
    feats = all_features[cmd][0]   # (n_frames, 39)
    im = axes[idx].imshow(feats.T, aspect='auto', origin='lower',
                           cmap='RdBu_r', interpolation='nearest')
    axes[idx].set_title(f'"{cmd}" — {feats.shape[0]} frames')
    axes[idx].set_xlabel('Frame index')
    axes[idx].set_ylabel('MFCC coefficient index (0–38)')
    plt.colorbar(im, ax=axes[idx])

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'Notebooks', 'Command_Recognition', 'mfcc_features.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 4. 4️⃣ GMM Training

**How GMM works for speech commands:**
1. All MFCC frames from recordings of `"open"` are stacked → training data for GMM-open
2. The GMM learns K Gaussian components that fit the acoustic distribution of that word
3. At inference: compute log P(frames | GMM-open) and log P(frames | GMM-status)
4. The command with **higher total log-likelihood** wins

**GMM formula:**
$$P(x) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(x \mid \mu_k, \Sigma_k)$$

Where $\pi_k$ = mixture weight, $\mu_k$ = mean, $\Sigma_k$ = covariance of component $k$.


In [ ]:
# ─── Train one GMM per command ────────────────────────────────────────────────
GMM_COMPONENTS = 4
GMM_COV_TYPE   = 'diag'   # diagonal = faster + avoids overfitting on small data

gmm_models = {}

print(f'Training GMM (n_components={GMM_COMPONENTS}, cov_type="{GMM_COV_TYPE}")...')
print('=' * 50)

for cmd in COMMANDS:
    if not all_features[cmd]:
        print(f'  [{cmd}] ⚠ No data — skipping')
        continue

    X = np.vstack(all_features[cmd])   # (total_frames, 39)
    print(f'  [{cmd}] Training on {X.shape[0]} frames ...')

    gmm = GaussianMixture(
        n_components=GMM_COMPONENTS,
        covariance_type=GMM_COV_TYPE,
        max_iter=200,
        n_init=5,
        random_state=42,
    )
    gmm.fit(X)

    # Save model
    model_path = os.path.join(MODELS_DIR, f'GMM_{cmd}.joblib')
    joblib.dump(gmm, model_path)
    gmm_models[cmd] = gmm

    score = gmm.score(X)
    converged = gmm.converged_
    print(f'  [{cmd}] ✅ Done — log-likelihood: {score:.4f} | converged: {converged}')
    print(f'          Saved → {model_path}')

print('\n✅ GMM training complete.')

In [ ]:
# ─── Visualize GMM: component means as heatmap ────────────────────────────────
if len(gmm_models) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('GMM Component Means (each row = one Gaussian component)', fontsize=13)

    for idx, cmd in enumerate(COMMANDS):
        if cmd not in gmm_models:
            continue
        means = gmm_models[cmd].means_   # (n_components, 39)
        im = axes[idx].imshow(means, aspect='auto', cmap='RdBu_r')
        axes[idx].set_title(f'GMM "{cmd}" — {means.shape[0]} components')
        axes[idx].set_xlabel('Feature dimension (39)')
        axes[idx].set_ylabel('GMM Component')
        axes[idx].set_yticks(range(means.shape[0]))
        axes[idx].set_yticklabels([f'G{i}' for i in range(means.shape[0])])
        plt.colorbar(im, ax=axes[idx])

    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'Notebooks', 'Command_Recognition', 'gmm_means.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

## 5. 5️⃣ HMM Training

**How HMM models speech commands:**

```
Left-to-right HMM (5 states for "open"):

→ [S1: onset] → [S2: vowel /o/] → [S3: /p/] → [S4: /ɛn/] → [S5: silence]
         ↑                                                           ↑
   self-loop                                                   absorbing state
```

**HMM parameters learned:**
- $\pi$ = initial state probabilities (starts at S1)
- $A$ = state transition matrix (left-to-right: can stay or advance)
- $B$ = emission probabilities — P(MFCC frame | state) → modeled as Gaussian

**Viterbi decoding** finds the most likely state sequence for a test utterance.


In [ ]:
hmm_models = {}

if HMM_AVAILABLE:
    from hmmlearn import hmm as hmmlib

    HMM_N_STATES = 5
    HMM_N_ITER   = 100

    print(f'Training HMM (n_states={HMM_N_STATES}, left-to-right, Gaussian emissions)...')
    print('=' * 55)

    for cmd in COMMANDS:
        if not all_features[cmd]:
            print(f'  [{cmd}] ⚠ No data — skipping')
            continue

        sequences = all_features[cmd]
        X         = np.vstack(sequences)
        lengths   = [len(s) for s in sequences]

        print(f'  [{cmd}] Training HMM on {len(sequences)} utterances, {X.shape[0]} frames ...')

        model = hmmlib.GaussianHMM(
            n_components=HMM_N_STATES,
            covariance_type='diag',
            n_iter=HMM_N_ITER,
            random_state=42,
        )

        # Initialize left-to-right transition matrix
        transmat = np.zeros((HMM_N_STATES, HMM_N_STATES))
        for i in range(HMM_N_STATES):
            if i < HMM_N_STATES - 1:
                transmat[i, i]   = 0.6
                transmat[i, i+1] = 0.4
            else:
                transmat[i, i] = 1.0

        model.transmat_  = transmat
        model.startprob_ = np.array([1.0] + [0.0] * (HMM_N_STATES - 1))

        try:
            model.fit(X, lengths)
            score = model.score(X, lengths) / max(X.shape[0], 1)
            print(f'  [{cmd}] ✅ Done — avg log-likelihood/frame: {score:.4f}')
        except Exception as e:
            print(f'  [{cmd}] ⚠ Training failed: {e}')
            continue

        model_path = os.path.join(MODELS_DIR, f'HMM_{cmd}.joblib')
        joblib.dump(model, model_path)
        hmm_models[cmd] = model
        print(f'          Saved → {model_path}')

    print('\n✅ HMM training complete.')
else:
    print('⚠ hmmlearn not available — skipping HMM training.')
    print('  Install with: pip install hmmlearn')

In [ ]:
# ─── Visualize HMM Transition Matrix ─────────────────────────────────────────
if hmm_models:
    fig, axes = plt.subplots(1, len(hmm_models), figsize=(12, 4))
    if len(hmm_models) == 1:
        axes = [axes]
    fig.suptitle('HMM Learned Transition Matrices', fontsize=13, fontweight='bold')

    for idx, (cmd, model) in enumerate(hmm_models.items()):
        im = axes[idx].imshow(model.transmat_, cmap='Blues', vmin=0, vmax=1)
        axes[idx].set_title(f'HMM "{cmd}"')
        axes[idx].set_xlabel('To state')
        axes[idx].set_ylabel('From state')
        for i in range(HMM_N_STATES):
            for j in range(HMM_N_STATES):
                axes[idx].text(j, i, f'{model.transmat_[i,j]:.2f}',
                               ha='center', va='center', fontsize=8,
                               color='white' if model.transmat_[i,j] > 0.5 else 'black')
        plt.colorbar(im, ax=axes[idx])

    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'Notebooks', 'Command_Recognition', 'hmm_transitions.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No HMM models to visualize.')

## 6. 6️⃣ Evaluation

### 7. 📊 Metrics for Biometric Command Recognition

| Metric | Formula | Why Use It? |
|--------|---------|-------------|
| **Accuracy** | (TP + TN) / N | Overall correctness |
| **Precision** | TP / (TP + FP) | How often accepted = correct |
| **Recall** | TP / (TP + FN) | How often correct = accepted |
| **F1-Score** | 2 × P × R / (P+R) | Balanced P & R |
| **WER** | (S + D + I) / N | Standard ASR metric |

### 8. 🔤 Word Error Rate (WER)

$$\text{WER} = \frac{S + D + I}{N}$$

Where:
- **S** = Substitutions (wrong word recognized)
- **D** = Deletions (word missed) 
- **I** = Insertions (extra word added)
- **N** = Total reference words

For **isolated-word** recognition (1 word per utterance):
- D = 0 (always 1 word out)
- I = 0 (always 1 word predicted)
- **WER = S / N = misclassification rate**

So for isolated commands: **WER = 1 − Accuracy**


In [ ]:
def compute_wer(y_true, y_pred):
    """Compute WER for isolated-word recognition."""
    n    = len(y_true)
    subs = sum(1 for t, p in zip(y_true, y_pred) if t != p)
    wer  = subs / n if n > 0 else 0.0
    return {
        'WER'          : round(wer, 4),
        'Accuracy'     : round(1 - wer, 4),
        'Substitutions': subs,
        'Deletions'    : 0,
        'Insertions'   : 0,
        'Total_Words'  : n,
    }


def evaluate_gmm(models, all_features, commands):
    """Evaluate GMM models using log-likelihood scoring."""
    y_true, y_pred = [], []
    scores_per_sample = []

    for true_cmd in commands:
        for feats in all_features[true_cmd]:
            cmd_scores = {cmd: models[cmd].score(feats)
                          for cmd in commands if cmd in models}
            predicted  = max(cmd_scores, key=cmd_scores.get)
            y_true.append(true_cmd)
            y_pred.append(predicted)
            scores_per_sample.append(cmd_scores)

    return y_true, y_pred, scores_per_sample


def evaluate_hmm(models, all_features, commands):
    """Evaluate HMM models using Viterbi log-likelihood."""
    y_true, y_pred = [], []

    for true_cmd in commands:
        for feats in all_features[true_cmd]:
            cmd_scores = {}
            for cmd, model in models.items():
                try:
                    s = model.score(feats) / max(len(feats), 1)
                except Exception:
                    s = -np.inf
                cmd_scores[cmd] = s
            predicted = max(cmd_scores, key=cmd_scores.get)
            y_true.append(true_cmd)
            y_pred.append(predicted)

    return y_true, y_pred


# Run GMM evaluation
print('Evaluating GMM models ...')
gt_gmm, pred_gmm, scores_gmm = evaluate_gmm(gmm_models, all_features, COMMANDS)
wer_gmm = compute_wer(gt_gmm, pred_gmm)

print(f'\nGMM Results:')
print(f'  Accuracy : {wer_gmm["Accuracy"]*100:.2f}%')
print(f'  WER      : {wer_gmm["WER"]*100:.2f}%')
print(f'  Errors   : {wer_gmm["Substitutions"]} / {wer_gmm["Total_Words"]}')

# Run HMM evaluation
gt_hmm, pred_hmm = [], []
wer_hmm = None

if hmm_models:
    print('\nEvaluating HMM models ...')
    gt_hmm, pred_hmm = evaluate_hmm(hmm_models, all_features, COMMANDS)
    wer_hmm = compute_wer(gt_hmm, pred_hmm)

    print(f'\nHMM Results:')
    print(f'  Accuracy : {wer_hmm["Accuracy"]*100:.2f}%')
    print(f'  WER      : {wer_hmm["WER"]*100:.2f}%')
    print(f'  Errors   : {wer_hmm["Substitutions"]} / {wer_hmm["Total_Words"]}')
else:
    print('\n⚠ HMM models not available for evaluation.')

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
n_models = 2 if hmm_models else 1
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5))
if n_models == 1:
    axes = [axes]

fig.suptitle('Confusion Matrices — Command Recognition', fontsize=14, fontweight='bold')

# GMM Confusion Matrix
cm_gmm = confusion_matrix(gt_gmm, pred_gmm, labels=COMMANDS)
sns.heatmap(cm_gmm, annot=True, fmt='d', cmap='Blues',
            xticklabels=COMMANDS, yticklabels=COMMANDS, ax=axes[0],
            linewidths=0.5, cbar=True)
axes[0].set_title(f'GMM — Accuracy: {wer_gmm["Accuracy"]*100:.1f}% | WER: {wer_gmm["WER"]*100:.1f}%')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# HMM Confusion Matrix
if hmm_models and wer_hmm:
    cm_hmm = confusion_matrix(gt_hmm, pred_hmm, labels=COMMANDS)
    sns.heatmap(cm_hmm, annot=True, fmt='d', cmap='Greens',
                xticklabels=COMMANDS, yticklabels=COMMANDS, ax=axes[1],
                linewidths=0.5, cbar=True)
    axes[1].set_title(f'HMM — Accuracy: {wer_hmm["Accuracy"]*100:.1f}% | WER: {wer_hmm["WER"]*100:.1f}%')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'Notebooks', 'Command_Recognition', 'confusion_matrices.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Full Classification Report ───────────────────────────────────────────────
print('=' * 55)
print('  GMM Classification Report')
print('=' * 55)
print(classification_report(gt_gmm, pred_gmm, target_names=COMMANDS))

if hmm_models and wer_hmm:
    print('=' * 55)
    print('  HMM Classification Report')
    print('=' * 55)
    print(classification_report(gt_hmm, pred_hmm, target_names=COMMANDS))

In [ ]:
# ─── WER Evaluation Table ─────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  Word Error Rate (WER) Evaluation Matrix')
print('=' * 60)

rows = []
row_gmm = {
    'Model'        : 'GMM',
    'Total Words'  : wer_gmm['Total_Words'],
    'Substitutions': wer_gmm['Substitutions'],
    'Deletions'    : wer_gmm['Deletions'],
    'Insertions'   : wer_gmm['Insertions'],
    'WER (%)'      : f"{wer_gmm['WER']*100:.2f}",
    'Accuracy (%)' : f"{wer_gmm['Accuracy']*100:.2f}",
}
rows.append(row_gmm)

if wer_hmm:
    row_hmm = {
        'Model'        : 'HMM',
        'Total Words'  : wer_hmm['Total_Words'],
        'Substitutions': wer_hmm['Substitutions'],
        'Deletions'    : wer_hmm['Deletions'],
        'Insertions'   : wer_hmm['Insertions'],
        'WER (%)'      : f"{wer_hmm['WER']*100:.2f}",
        'Accuracy (%)' : f"{wer_hmm['Accuracy']*100:.2f}",
    }
    rows.append(row_hmm)

wer_df = pd.DataFrame(rows)
print(wer_df.to_string(index=False))

# ─── WER per command breakdown ────────────────────────────────────────────────
print('\n  WER per Command (GMM):')
for cmd in COMMANDS:
    cmd_true  = [t for t, p in zip(gt_gmm, pred_gmm) if t == cmd]
    cmd_pred  = [p for t, p in zip(gt_gmm, pred_gmm) if t == cmd]
    cmd_wer   = compute_wer(cmd_true, cmd_pred)
    print(f'    "{cmd}": WER={cmd_wer["WER"]*100:.1f}% | '
          f'Acc={cmd_wer["Accuracy"]*100:.1f}% | '
          f'Errors={cmd_wer["Substitutions"]}/{cmd_wer["Total_Words"]}')

In [ ]:
# ─── Visualize WER comparison ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('GMM vs HMM — Performance Comparison', fontsize=13, fontweight='bold')

models_names = ['GMM']
accuracies   = [wer_gmm['Accuracy'] * 100]
wer_values   = [wer_gmm['WER'] * 100]
bar_colors   = ['#2196F3']

if wer_hmm:
    models_names.append('HMM')
    accuracies.append(wer_hmm['Accuracy'] * 100)
    wer_values.append(wer_hmm['WER'] * 100)
    bar_colors.append('#4CAF50')

# Accuracy bar chart
bars1 = ax1.bar(models_names, accuracies, color=bar_colors, edgecolor='black', linewidth=0.8)
ax1.set_title('Accuracy (%)')
ax1.set_ylabel('Accuracy (%)')
ax1.set_ylim([0, 110])
for bar, val in zip(bars1, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.4)

# WER bar chart
bars2 = ax2.bar(models_names, wer_values, color=bar_colors, edgecolor='black', linewidth=0.8)
ax2.set_title('Word Error Rate — WER (%)')
ax2.set_ylabel('WER (%)')
ax2.set_ylim([0, max(wer_values) * 1.3 + 1])
for bar, val in zip(bars2, wer_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'Notebooks', 'Command_Recognition', 'gmm_vs_hmm.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 9. 7️⃣ Permission-Based Command System

After verification, commands are gated by user identity:

| User | Verification | Allowed Commands |
|------|-------------|------------------|
| `User_01` | ✅ Admin | `open`, `status` |
| `User_02` | ✅ Regular | `status` only |
| Unknown | ❌ Unverified | None |


In [ ]:
USER_PERMISSIONS = {
    'User_01': ['open', 'status'],
    'User_02': ['status'],
}


def get_allowed_commands(user_id):
    return USER_PERMISSIONS.get(user_id, [])


def check_permission(user_id, command):
    if command == 'unknown':
        return {'allowed': False, 'reason': '❓ Command not recognized'}
    allowed = get_allowed_commands(user_id)
    if not allowed:
        return {'allowed': False, 'reason': '🚫 User not verified'}
    if command in allowed:
        return {'allowed': True, 'reason': f'✅ "{command}" executed'}
    return {'allowed': False,
            'reason': f'⚠️ Permission Denied: "{command}" requires Admin (User_01)'}


# ─── Demo permission table ────────────────────────────────────────────────────
print('=' * 60)
print('  Permission System Demo')
print('=' * 60)

test_cases = [
    ('User_01', 'open'),
    ('User_01', 'status'),
    ('User_02', 'status'),
    ('User_02', 'open'),
    ('Unknown',  'status'),
    ('Unknown',  'open'),
]

for user, cmd in test_cases:
    result = check_permission(user, cmd)
    status = '✅ ALLOWED' if result['allowed'] else '❌ DENIED'
    print(f'  {user:<10} + "{cmd:<6}" → {status} | {result["reason"]}')

## 10. 8️⃣ End-to-End Demo

Full pipeline: **Audio → Features → GMM/HMM Predict → Permission Check → Execute**


In [ ]:
def predict_command_full(audio_path, prefer='GMM'):
    """Full prediction pipeline."""
    feats = extract_mfcc_sequence(audio_path)

    if prefer == 'GMM' and gmm_models:
        cmd_scores = {cmd: gmm_models[cmd].score(feats) for cmd in COMMANDS if cmd in gmm_models}
        best_cmd   = max(cmd_scores, key=cmd_scores.get)
        return {'command': best_cmd, 'scores': cmd_scores, 'model': 'GMM'}

    elif prefer == 'HMM' and hmm_models:
        cmd_scores = {}
        for cmd, m in hmm_models.items():
            try:
                cmd_scores[cmd] = m.score(feats) / max(len(feats), 1)
            except Exception:
                cmd_scores[cmd] = -np.inf
        best_cmd = max(cmd_scores, key=cmd_scores.get)
        return {'command': best_cmd, 'scores': cmd_scores, 'model': 'HMM'}

    return {'command': 'unknown', 'scores': {}, 'model': 'none'}


# ─── Run demo on test samples ─────────────────────────────────────────────────
print('=' * 60)
print('  End-to-End Demo — Command Recognition Pipeline')
print('=' * 60)

test_scenarios = [
    ('User_01', 'open',   'open'),
    ('User_02', 'status', 'status'),
    ('User_02', 'open',   'open'),     # Will be DENIED
]

for user, expected_cmd, actual_cmd_folder in test_scenarios:
    cmd_path  = os.path.join(COMMANDS_DIR, actual_cmd_folder)
    wav_files = sorted([f for f in os.listdir(cmd_path) if f.lower().endswith(AUDIO_EXTENSIONS)])

    if not wav_files:
        print(f'  No files for "{actual_cmd_folder}" — skipping')
        continue

    test_file = os.path.join(cmd_path, wav_files[-1])  # use last file
    result    = predict_command_full(test_file, prefer='GMM')
    perm      = check_permission(user, result['command'])

    print(f'\n  User: {user} | Expected: "{expected_cmd}" | Predicted: "{result["command"]}"')
    print(f'  Scores: {result["scores"]}')
    print(f'  Permission: {"✅ ALLOWED" if perm["allowed"] else "❌ DENIED"} → {perm["reason"]}')

## 11. 9️⃣ Summary

| Aspect | GMM | HMM |
|--------|-----|-----|
| **Models speech as** | Static distribution of MFCC frames | Sequential states over time |
| **Temporal modeling** | ❌ No (frames pooled) | ✅ Yes (left-to-right states) |
| **Training data needed** | Less (all frames pooled) | More (needs sequence structure) |
| **Inference** | log-likelihood scoring | Viterbi decoding |
| **Best for** | Short, stable sounds | Words with temporal dynamics |
| **WER (this dataset)** | See above | See above |

### 12. Key Takeaways:
- **WER = 1 − Accuracy** for isolated-word recognition (no D or I)
- **GMM** is simpler and fast; works well when commands are acoustically distinct
- **HMM** captures the temporal evolution of pronunciation — better for longer or more complex commands
- **Permission system** ensures verified users only get access to their allowed commands
- Integration in `src/apps/live_camera_app.py` activates command recognition AFTER `ACCESS GRANTED`


In [ ]:
# ─── Final Summary ────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  Command Recognition — Final Summary')
print('=' * 60)
print(f'  Commands : {COMMANDS}')
print(f'  GMM Acc  : {wer_gmm["Accuracy"]*100:.2f}% | WER: {wer_gmm["WER"]*100:.2f}%')
if wer_hmm:
    print(f'  HMM Acc  : {wer_hmm["Accuracy"]*100:.2f}% | WER: {wer_hmm["WER"]*100:.2f}%')
print()
print('  Saved Models:')
for f in os.listdir(MODELS_DIR):
    fpath = os.path.join(MODELS_DIR, f)
    size  = os.path.getsize(fpath) / 1024
    print(f'    {f} ({size:.1f} KB)')
print()
print('  User Permissions:')
for user, cmds in USER_PERMISSIONS.items():
    print(f'    {user}: {cmds}')
print('=' * 60)